# 第 9 章 決定木

「どの質問をすれば、もっともよくデータが分かれるか」を貪欲に選び続けて木を育てます。

対応する記事: [第 9 章 決定木（Kotlin 版）](https://github.com/k2works/grokking-machine-learning-excersice/blob/main/docs/article/grokking-machine-learning/kotlin/ch09.md)

実装本体: `apps/grokking-ml-kotlin/src/`

## セットアップ

実装本体をビルドした JAR を読み込みます。**ノートブックにコードを複製せず、記事と同じ実装をそのまま使います。**

先に JAR を作っておいてください。

```bash
cd apps/grokking-ml-kotlin
./gradlew jar
```

IntelliJ IDEA の Kotlin Notebook プラグイン、または [Kotlin Jupyter カーネル](https://github.com/Kotlin/kotlin-jupyter) で開きます。

```bash
pip install kotlin-jupyter-kernel
jupyter lab notebooks/
```

In [1]:
@file:DependsOn("../build/libs/grokking-ml-kotlin-0.1.0.jar")

import ch09.*

## 不純度

**ジニ不純度は「ランダムに 2 つ選んだとき、ラベルが食い違う確率」**、**エントロピーは「ラベルを 1 つ伝えるのに必要なビット数」** と読めます。値の範囲は違いますが、順序は一致します。

In [2]:
println("%-20s %8s %14s".format("集合", "ジニ", "エントロピー"))
listOf("純粋 [1,1,1]" to listOf(1, 1, 1),
       "偏り [1,1,1,0]" to listOf(1, 1, 1, 0),
       "均等 [1,1,0,0]" to listOf(1, 1, 0, 0),
       "4 クラス均等" to listOf(0, 1, 2, 3)).forEach { (name, ls) ->
    println("%-20s %8.4f %14.4f".format(name, giniImpurity(ls), entropy(ls)))
}

集合                         ジニ         エントロピー


純粋 [1,1,1]             0.0000        -0.0000
偏り [1,1,1,0]           0.3750         0.8113


均等 [1,1,0,0]           0.5000         1.0000


4 クラス均等                0.7500         2.0000


## データセット

原著と同じアプリ推薦データです。特徴量は性別（0=女性、1=男性）と年齢。**「若い人には推薦する」という規則が隠れていますが、性別は関係ありません。**

In [3]:
val points = listOf(listOf(1.0, 15.0), listOf(0.0, 25.0), listOf(0.0, 32.0), listOf(1.0, 35.0),
                    listOf(0.0, 12.0), listOf(1.0, 14.0), listOf(1.0, 55.0), listOf(0.0, 40.0))
val labels = listOf(1, 0, 0, 0, 1, 1, 0, 0)

println("根の不純度 ジニ %.4f / エントロピー %.4f".format(giniImpurity(labels), entropy(labels)))

根の不純度 ジニ 0.4688 / エントロピー 0.9544


## すべての分割候補を評価する

**決定木は自力で「年齢が効く、性別は関係ない」を見つけます。** 「年齢 < 20」の利得が根の不純度と一致し、この 1 回の質問で不純度が 0 になることが分かります。

In [4]:
println("%-8s %8s %10s".format("特徴量", "閾値", "情報利得"))
candidateSplits(points).forEach { split ->
    val partition = applySplit(points, labels, split)
    if (partition.leftLabels.isNotEmpty() && partition.rightLabels.isNotEmpty()) {
        val name = if (split.feature == 0) "性別" else "年齢"
        println("%-8s %8.1f %10.4f".format(name, split.threshold,
                informationGain(labels, partition.leftLabels, partition.rightLabels)))
    }
}

特徴量            閾値       情報利得
性別            0.5     0.0313


年齢           13.0     0.1116
年齢           14.5     0.2604
年齢           20.0     0.4688


年齢           28.5     0.2813
年齢           33.5     0.1688
年齢           37.5     0.0938


年齢           47.5     0.0402


## 木を育てる

**深さ 1、葉 2 枚の木で正解率 1.0 に達します。** モデルをそのまま出力して読めるのが決定木の強みです。

In [5]:
val tree = buildTree(points, labels)

println(tree)
println("深さ %d  葉の数 %d  正解率 %.2f".format(depth(tree), leafCount(tree), accuracy(tree, points, labels)))

Node(split=Split(feature=1, threshold=20.0), left=Leaf(label=1), right=Leaf(label=0))


深さ 1  葉の数 2  正解率 1.00


## ジニとエントロピーは同じ木を作る

値の絶対値は違う（0.4688 と 0.9544）のに、**順序が同じなので選ばれる分割も同じ** になります。

In [6]:
val byGini = buildTree(points, labels, impurity = giniImpurity)
val byEntropy = buildTree(points, labels, impurity = entropy)
println("同じ木か: " + (byGini == byEntropy))

同じ木か: true


## 試してみる: 木の成長を止める

最大深さと最小サンプル数は、**第 4 章の正則化と同じ役割** です。木は放っておくと訓練データを丸暗記するまで育ちます。

In [7]:
listOf(0, 1, 3).forEach { maxDepth ->
    val t = buildTree(points, labels, maxDepth = maxDepth)
    println("maxDepth=%d  深さ %d  葉 %d  正解率 %.2f".format(maxDepth, depth(t), leafCount(t),
            accuracy(t, points, labels)))
}

maxDepth=0  深さ 0  葉 1  正解率 0.63
maxDepth=1  深さ 1  葉 2  正解率 1.00


maxDepth=3  深さ 1  葉 2  正解率 1.00
